# AIE425 Assignment 1 — Member A
## Ahmed Essa | 223106563 | Track A + assigned shared work

This notebook implements S1 P1 (rating statistics), S3 P3 (complexity), shared data preparation, and the item-based CF baseline assigned to Member A.

**Missing ratings are not zero.** The 80/20 interaction split uses seed 425. Place MovieLens 1M `ratings.dat` in `../data/raw/` before running.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from irs_assignment.data import load_ratings, split_ratings, build_sparse_matrix, dataset_summary, SEED
from irs_assignment.statistics import rating_statistics
from irs_assignment.item_based import evaluate_item_based
from irs_assignment.complexity import complexity_table, dense_vs_sparse_bytes
DATA_PATH = ROOT / 'data' / 'raw' / 'ratings.dat'
ratings = load_ratings(DATA_PATH)
print('Rows:', ratings.height)
print('Columns:', ratings.columns)
ratings.head()


## 1. Dataset overview and validation
We inspect dataset dimensions, rating range, unique users/items, density, and sparsity.

In [ ]:
summary = dataset_summary(ratings)
summary


## 2. S1 P1 — Rating statistics
The next table shows overall descriptive statistics. The rating-frequency table shows both counts and percentages.

In [ ]:
stats = rating_statistics(ratings)
stats['overall']


In [ ]:
stats['frequency']


In [ ]:
freq = stats['frequency'].to_pandas()
plt.figure(figsize=(7,4))
plt.bar(freq['rating'].astype(str), freq['len'])
plt.title('MovieLens 1M: Rating Frequency')
plt.xlabel('Rating (stars)'); plt.ylabel('Number of ratings'); plt.tight_layout(); plt.show()


**Interpretation prompt:** Which rating value is most frequent? Is the distribution skewed toward positive ratings? Use the actual frequency percentages above to support the answer.

In [ ]:
stats['user_activity_summary'], stats['item_popularity_summary']


In [ ]:
user_means = stats['per_user']['mean_rating'].to_numpy()
plt.figure(figsize=(7,4)); plt.hist(user_means, bins=30)
plt.title('Distribution of Average Rating per User'); plt.xlabel('Mean user rating'); plt.ylabel('Number of users'); plt.tight_layout(); plt.show()
item_counts = stats['per_item']['rating_count'].to_numpy()
plt.figure(figsize=(7,4)); plt.hist(item_counts, bins=40)
plt.title('Distribution of Ratings per Movie'); plt.xlabel('Ratings per movie'); plt.ylabel('Number of movies'); plt.tight_layout(); plt.show()


## 3. Shared preprocessing — seeded 80/20 split and sparse matrix
The split is interaction-level, not user-level. The full user/item ID universe is retained when building the training CSR matrix.

In [ ]:
train, test = split_ratings(ratings, test_fraction=0.20, seed=SEED)
all_users = sorted(ratings['user_id'].unique().to_list())
all_items = sorted(ratings['item_id'].unique().to_list())
train_matrix, user_map, item_map = build_sparse_matrix(train, all_users, all_items)
print('Train:', train.height, 'Test:', test.height)
print('CSR shape:', train_matrix.shape, 'Stored ratings:', train_matrix.nnz)
print('Density:', train_matrix.nnz / (train_matrix.shape[0] * train_matrix.shape[1]))


## 4. Shared item-based collaborative filtering baseline
Adjusted cosine is computed over co-rated users only. User means are estimated from training ratings. Undefined similarities remain NaN.

In [ ]:
test_rows = np.array([user_map[x] for x in test['user_id'].to_list()], dtype=np.int32)
test_cols = np.array([item_map[x] for x in test['item_id'].to_list()], dtype=np.int32)
test_values = test['rating'].to_numpy().astype(float)
result = evaluate_item_based(train_matrix, test_rows, test_cols, test_values, k=20)
{key: result[key] for key in ['mae','rmse','coverage','n_test','n_predicted']}


## 5. S3 P3 — Computational complexity
Let U be users, I items, E observed ratings, T test interactions, and k the number of neighbors. Sparse matrix products have data-dependent cost; the dense item-item similarity output requires O(I²) storage.

In [ ]:
complexity = complexity_table(len(all_users), len(all_items), ratings.height, 20)
pl.DataFrame(complexity)


In [ ]:
dense_vs_sparse_bytes(len(all_users), len(all_items), ratings.height)


## 6. Summary and discussion
Complete the written discussion using the outputs above. Compare item-based MAE/RMSE/coverage with the group's user-based CF results only after both use the same train/test split and metric definitions.

**Important:** This notebook does not implement Maryam's Track B or Ahmed Allam's Track C tasks. Confirm final conventions with the group before submission.
